# Helper Scripts

A scratch workbook of one-off helpers used during dataset assembly and
evaluation. Cells cover:

- fixing logic forms (e.g., replacing `eat(...)` with `drink(...)` for beverages),
- filtering and re-indexing story IDs (removing selected SIDs, renumbering to a
  continuous `0..n` sequence),
- archiving and converting `encodedForm/fixed_all_combined.json` (JSON <-> XML),
- generating few-shot input files (`experiments/FewShot/inputs/`).

See `RECONSTRUCTION.md` for the end-to-end dataset pipeline.

In [ ]:
# import json
# import re
# from pathlib import Path
# import os

# # Function to fix eat(...) to drink(...) for beverages
# # not to be run everytime
# def fix_eat_to_drink(logic_str):
#     # Step 1: Find all beverage items
#     beverage_pattern = r'beverage\("([^"]+)"\)'
#     beverages = set(re.findall(beverage_pattern, logic_str))

#     if not beverages:
#         return logic_str  # No beverages → nothing to change

#     # Step 2: For each beverage, replace eat(..., "beverage") with drink(...)
#     updated_str = logic_str
#     for bev in beverages:
#         # Escape special regex chars in beverage name (though unlikely in your data)
#         escaped_bev = re.escape(bev)
#         # Match eat( <anything> , "bev" )
#         # We assume the second argument is the food/drink item
#         eat_pattern = r'\beat\(([^,)]+),\s*"{}"\)'.format(escaped_bev)
#         drink_repl = r'drink(\1, "{}")'.format(bev)
#         updated_str = re.sub(eat_pattern, drink_repl, updated_str)

#     return updated_str

# # Configuration - resolve paths relative to current working directory
# base_dir = Path(os.getcwd())
# input_file = base_dir.parent / "encodedForm" / "all_combined.json"
# output_file = base_dir.parent / "encodedForm" / "fixed_all_combined.json"

# # Load your JSON file
# with open(input_file, 'r') as f:
#     data = json.load(f)

# # Process each entry in the "data" array
# for item in data['data']:
#     logic_list = item['logic_form']
#     # logic_form is a list with one string
#     if logic_list:
#         original = logic_list[0]
#         fixed = fix_eat_to_drink(original)
#         item['logic_form'][0] = fixed

# # Optional: Save the updated JSON
# with open(output_file, 'w') as f:
#     json.dump(data, f, indent=2)

# print("Processing complete.")

Processing complete.


In [6]:
# check if any logic_form has beverage but no drink action

import json
import re
from pathlib import Path
import os

# Configuration - resolve paths relative to current working directory
base_dir = Path(os.getcwd())
input_file = base_dir.parent / "encodedForm" / "fixed_all_combined.json"
# output_file = base_dir.parent / "encodedForm" / "fixed_all_combined.json"

# Load your JSON file
with open(input_file, 'r') as f:
    data = json.load(f)

# Iterate over each story object
for item in data['data']:
    logic_str = item['logic_form'][0]  # it's a list with one string

    has_beverage = 'beverage(' in logic_str
    has_drink = 'drink(' in logic_str

    if has_beverage and not has_drink:
        print(item['sid'])

42
49
56
61
64
74
76
78
91


In [1]:
# create an archive of the fixed_all_combined.json file
import shutil
from datetime import datetime
import os

from pathlib import Path

# Configuration - resolve paths relative to current working directory
base_dir = Path(os.getcwd())
input_file = base_dir.parent / "encodedForm" / "fixed_all_combined.json"
# output_file = base_dir.parent / "encodedForm" / "fixed_all_combined.json"

def backup_json_file(filepath):
    # Ensure the file exists
    if not os.path.isfile(filepath):
        raise FileNotFoundError(f"File {filepath} does not exist.")
    
    # Create a timestamped backup name
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = f"{filepath}.backup_{timestamp}"
    
    # Copy the file
    shutil.copy2(filepath, backup_path)
    print(f"Backup created: {backup_path}")
    return backup_path

# Example usage
original_file = input_file
backup_json_file(original_file)



Backup created: encodedForm/fixed_all_combined.json.backup_20251124_124855


'encodedForm/fixed_all_combined.json.backup_20251124_124855'

In [ ]:
# sids that need to be filtered out:
filtered_out_sids = [56, 58, 64, 71, 72, 87, 89, 96, 103, 108, 110, 111]

# print(len(filtered_out_sids))

# remove entries with these sids from fixed_all_combined.json
# first, load the json file
import json
with open(input_file, 'r') as f:
    data = json.load(f)


# filter out the entries
data['data'] = [item for item in data['data'] if item['sid'] not in filtered_out_sids]
with open(input_file, 'w') as f:
    json.dump(data, f, indent=2)


In [9]:
# fix the order of sids, and make it a continuous sequence without gaps in fixed_all_combined.json
import json
import os
import shutil
from pathlib import Path
from datetime import datetime
from operator import itemgetter

# Configuration - resolve paths relative to current working directory
base_dir = Path(os.getcwd())
input_file = base_dir.parent / "encodedForm" / "fixed_all_combined.json"
# output_file = base_dir.parent / "encodedForm" / "fixed_all_combined.json"

# load the json file
with open(input_file, 'r') as f:
    data = json.load(f)

# sort the data by sid
data['data'] = sorted(data['data'], key=itemgetter('sid'))

# reassign sids to be continuous
for new_sid, item in enumerate(data['data']):
    item['sid'] = new_sid

# save the updated json file
with open(input_file, 'w') as f:
    json.dump(data, f, indent=2)

print("SID values reindexed to 0, 1, 2, ...", len(data['data']))



SID values reindexed to 0, 1, 2, ... 100


In [10]:
# Conversion from JSON to XML with formatted logic_form

import json
import xmltodict
from pathlib import Path
import os

# input and output file paths
base_dir = Path(os.getcwd())
input_file = base_dir.parent / "encodedForm" / "fixed_all_combined.json"
output_file = base_dir.parent / "encodedForm" / "fixed_all_combined.xml"

# 1. Load the JSON file
with open(input_file, 'r') as f:
    json_content = json.load(f)

# 2. Pre-process the data to format 'logic_form'
# We iterate through every story in the data list
for story in json_content['data']:
    if 'logic_form' in story and len(story['logic_form']) > 0:
        # Get the raw string (it is currently inside a list like ["..."])
        raw_logic = story['logic_form'][0]
        
        # REFORMATTING MAGIC:
        # 1. Replace ". " with ".\n\t\t\t" (New line + 3 tabs for indentation)
        # 2. Add a newline at the very start so the first item isn't on the same line as the tag
        formatted_logic = "\n\t\t\t" + raw_logic.replace(". ", ".\n\t\t\t")
        
        # Update the dictionary with the new string
        # We remove it from the list so it becomes a direct text node in XML
        story['logic_form'] = formatted_logic

# 3. Wrap in a root element (Required for valid XML)
wrapped_content = {'dataset': json_content}

# 4. Convert and Save
with open(output_file, 'w') as f:
    # 'pretty=True' handles the normal tag indentation
    xmltodict.unparse(wrapped_content, output=f, pretty=True)

print("Conversion complete! Check output_formatted.xml")

Conversion complete! Check output_formatted.xml


In [1]:
# Keep the logic form of only first 5 stories, delete all other and make it N/A and 
# store the new file in experiments/FewShot/5_shot_input.json
import json
from pathlib import Path
import os

# input and output file paths
base_dir = Path(os.getcwd())
input_file = base_dir.parent / "encodedForm" / "fixed_all_combined.json"
output_file = base_dir.parent / "experiments" / "FewShot" / "fewshot_5_input.json"

# Ensure output directory exists
output_file.parent.mkdir(parents=True, exist_ok=True)

# Load the input JSON and only keep the 'data' part
with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)
    data = {'data': data['data']}
    print(f"Total stories loaded: {len(data['data'])}")


# Process the "data" list
for i, story in enumerate(data["data"]):
    if i < 5:
        # Keep logic_form as is (ensure it's a list)
        pass
    else:
        # Replace logic_form with ["N/A"]
        story["logic_form"] = ["N/A"]

# Write the modified data to the output file
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)
print(f"Processed data saved to {output_file}")


Total stories loaded: 100
Processed data saved to experiments/FewShot/fewshot_5_input.json


In [3]:
# Keep the logic form of only 5 random stories, delete all other and make it N/A and 
# store the new file in experiments/FewShot/5_random_input.json
import json
from pathlib import Path
import os

# input and output file paths
base_dir = Path(os.getcwd())
input_file = base_dir.parent / "encodedForm" / "fixed_all_combined.json"
output_file = base_dir.parent / "experiments" / "FewShot" / "inputs" / "5_random_input.json"

# Ensure output directory exists
output_file.parent.mkdir(parents=True, exist_ok=True)

# Load the input JSON and only keep the 'data' part
with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)
    stories = data['data']
    print(f"Total stories loaded: {len(stories)}")


# Randomly select 5 story indices
import random
if len(stories) < 5:
    raise ValueError("Not enough stories to sample 5.")
selected_indices = set(random.sample(range(len(stories)), 5))

# Process the "data" list: keep only 5 random stories' logic_form
for i, story in enumerate(stories):
    if i not in selected_indices:
        # Replace logic_form with ["N/A"]
        story["logic_form"] = ["N/A"]


# Write the modified data to the output file
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump({'data': stories}, f, indent=2, ensure_ascii=False)
print(f"Processed data saved to {output_file}")


Total stories loaded: 100
Processed data saved to experiments/FewShot/inputs/5_random_input.json


In [1]:
# random 10 stories. Everything else is same as above
# Keep the logic form of only 5 random stories, delete all other and make it N/A and 
# store the new file in experiments/FewShot/5_random_input.json
import json
from pathlib import Path
import os

# input and output file paths
base_dir = Path(os.getcwd())
input_file = base_dir.parent / "encodedForm" / "fixed_all_combined.json"
output_file = base_dir.parent / "experiments" / "FewShot" / "inputs" / "10_random_input.json"

# Ensure output directory exists
output_file.parent.mkdir(parents=True, exist_ok=True)

# Load the input JSON and only keep the 'data' part
with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)
    stories = data['data']
    print(f"Total stories loaded: {len(stories)}")


# Randomly select 10 story indices
import random
if len(stories) < 10:
    raise ValueError("Not enough stories to sample 10.")
selected_indices = set(random.sample(range(len(stories)), 10))

# Process the "data" list: keep only 10 random stories' logic_form
for i, story in enumerate(stories):
    if i not in selected_indices:
        # Replace logic_form with ["N/A"]
        story["logic_form"] = ["N/A"]


# Write the modified data to the output file
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump({'data': stories}, f, indent=2, ensure_ascii=False)
print(f"Processed data saved to {output_file}")


Total stories loaded: 100
Processed data saved to experiments/FewShot/inputs/10_random_input.json


In [1]:
# Remove the logic form from all the stories, make it N/A and
# store the new file in experiments/FewShot/0_input.json
import json
from pathlib import Path
import os

base_dir = Path(os.getcwd())
input_file = base_dir.parent / "encodedForm" / "fixed_all_combined.json"
output_file = base_dir.parent / "experiments" / "FewShot" / "inputs" / "0_input.json"

output_file.parent.mkdir(parents=True, exist_ok=True)

with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)
    stories = data['data']
    print(f"Total stories loaded: {len(stories)}")

for story in stories:
    story["logic_form"] = ["N/A"]

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump({'data': stories}, f, indent=2, ensure_ascii=False)
print(f"Saved to {output_file}")


Total stories loaded: 100
Saved to experiments/FewShot/inputs/0_input.json
